# NASA EONET Natural Disaster Analysis

**Authors:** Ernest Afful, Daniel Foulen


**Data Source:** NASA Earth Observatory Natural Event Tracker (EONET) v3  
**Coverage:** United States | January 2025 – March 2026

## Setup and Run Mode

Create a `.env` file in the same directory as this notebook:

| Label | Value |
|---|---|
| `NASA_API_KEY` | Your NASA EONET API key |
| `MAPBOX_TOKEN` | Your Mapbox public token |

- **Full run:** No cached data found. Fetches from EONET, saves raw and cleaned CSVs.
- **Staging mode:** CSVs found. API is not called. Geocoding is also skipped if cached.

In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import requests
import ast
import ssl
import certifi
import time
import os
from geopy.geocoders import Nominatim
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
APIKEY       = os.getenv("NASA_API_KEY")
MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN")

pio.renderers.default = "notebook"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RAW_CSV      = DATA_DIR / "nasa_eonet_raw.csv"
CLEANED_CSV  = DATA_DIR / "nasa_eonet_cleaned.csv"
GEOCODED_CSV = DATA_DIR / "nasa_eonet_geocoded.csv"

FULL_RUN = not RAW_CSV.exists()

f"Mode: {'Full run' if FULL_RUN else 'Staging (cached data found)'}"

'Mode: Staging (cached data found)'

## Data Acquisition

Fetches up to 10,000 events from the EONET v3 API across all statuses and categories.  
Skipped in staging mode.

In [24]:
if FULL_RUN:
    url = "https://eonet.gsfc.nasa.gov/api/v3/events"
    params = {
        "api_key": APIKEY,
        "limit": 10000,
        "status": "all",
        "start": "2025-01-01",
        "end":   "2026-03-31",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    events = response.json().get("events", [])
    df_raw = pd.DataFrame(events)
    df_raw.to_csv(RAW_CSV, index=False)
else:
    df_raw = pd.read_csv(RAW_CSV)

f"{len(df_raw)} raw events loaded"

'7101 raw events loaded'

## Cleaning

Steps:
1. Parse nested `geometry` and `categories` columns (stored as strings in CSV)
2. Extract latitude, longitude, date, and event type
3. Drop source columns not used in analysis
4. Filter to US bounding box (lat 18.9–71.5, lon -168.0 to -66.9)
5. Remove duplicates and rows missing key fields
6. Parse dates

In [25]:
if not CLEANED_CSV.exists():
    df = pd.read_csv(RAW_CSV)

    def parse_col(x):
        if isinstance(x, list):
            return x
        try:
            return ast.literal_eval(str(x))
        except Exception:
            return None

    df["_geo"]  = df["geometry"].apply(parse_col)
    df["_cats"] = df["categories"].apply(parse_col)

    df["latitude"]   = df["_geo"].apply(lambda x: x[0]["coordinates"][1] if x else None)
    df["longitude"]  = df["_geo"].apply(lambda x: x[0]["coordinates"][0] if x else None)
    df["date"]       = df["_geo"].apply(lambda x: x[0]["date"]            if x else None)
    df["event_type"] = df["_cats"].apply(lambda x: x[0]["title"]          if x else None)

    df = df.drop(columns=["geometry", "categories", "sources", "_geo", "_cats"])
    df = df.drop_duplicates()
    df = df.dropna(subset=["latitude", "longitude", "date", "event_type"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Filter to US bounding box (contiguous + Alaska + Hawaii)
    df = df[
        df["latitude"].between(18.9, 71.5) &
        df["longitude"].between(-168.0, -66.9)
    ].copy()

    df.to_csv(CLEANED_CSV, index=False)

else:
    df = pd.read_csv(CLEANED_CSV)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

df.head()

,id,title,description,link,closed,latitude,longitude,date,event_type
0,EONET_19229,"Julie Pond Wildfire, Dorchester, Maryland","12 Miles S from Cambridge, MD",https://eonet.gsfc.nasa.gov/api/v3/events/EONE...,NaN,38.396085,-75.976250,2026-03-31 13:22:00+00:00,Wildfires
1,EONET_19232,"Caddo Lake C-6 RX Prescribed Fire, Harrison, T...",NaN,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...,NaN,32.667308,-94.104296,2026-03-31 10:13:00+00:00,Wildfires
2,EONET_19224,"C-148 BLK 9 10 &amp; 11 EN RX Prescribed Fire,...",NaN,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...,NaN,34.421667,-81.528333,2026-03-31 07:55:00+00:00,Wildfires
3,EONET_19225,"Middle Toe Wildfire, Pittsburg, Oklahoma","7 Miles S from Quinton, OK",https://eonet.gsfc.nasa.gov/api/v3/events/EONE...,NaN,35.047500,-95.397778,2026-03-30 19:32:00+00:00,Wildfires
4,EONET_19194,"Hall Thompson Lane Wildfire, Shelby, Alabama","19 Miles N from Westover, AL",https://eonet.gsfc.nasa.gov/api/v3/events/EONE...,NaN,33.407778,-86.540833,2026-03-30 15:26:00+00:00,Wildfires


## Q1: Where do natural disasters cluster most densely in the US?

In [30]:
df["count"] = 1

px.set_mapbox_access_token(MAPBOX_TOKEN)

fig = px.density_map(
    df,
    lat="latitude", lon="longitude", z="count",
    radius=15,
    center=dict(lat=39.5, lon=-98.35),
    zoom=3,
    map_style="open-street-map",
    title="US Distribution of NASA EONET Events"
)
fig.show()

## Q2: Is the frequency of a specific event type increasing over time?

> **Note on severe storms:** Storms appear underrepresented. EONET aggregates from specific source agencies (USGS, GDACs, InciWeb) and storm-tracking agencies like NOAA/NWS do not always feed into EONET. This is a data pipeline limitation, not a reflection of true storm frequency.

In [27]:
df["month"] = df["date"].dt.month

monthly_counts = (
    df.groupby(["month", "event_type"])["id"]
    .count()
    .reset_index()
)

fig = px.line(
    monthly_counts,
    x="month", y="id", color="event_type",
    title="Frequency of Event Types by Month"
)
fig.show()

## Q3: Which months or seasons see the highest event activity?

In [28]:
fig = px.bar(
    monthly_counts,
    x="month", y="id", color="event_type",
    barmode="group",
    title="Monthly Event Activity by Type",
    labels={"id": "Number of Events", "month": "Month"}
)
fig.show()

## Q4: Which US regions are most prone to wildfires?

Coordinates are rounded to one decimal place and grouped to reduce geocoding volume.  
Top 50 hotspots are reverse geocoded via Nominatim (1 request/second per usage policy).  
State is extracted from the returned address string.

In [29]:
GEOCODED_CSV = DATA_DIR / "nasa_eonet_geocoded.csv"

df_wildfires = df[df["event_type"] == "Wildfires"].copy()

df_wildfires["lat_rounded"] = df_wildfires["latitude"].round(1)
df_wildfires["lon_rounded"] = df_wildfires["longitude"].round(1)

top_coords = (
    df_wildfires
    .groupby(["lat_rounded", "lon_rounded"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

if GEOCODED_CSV.exists():
    top_coords = pd.read_csv(GEOCODED_CSV)
else:
    ctx = ssl.create_default_context(cafile=certifi.where())
    geolocator = Nominatim(user_agent="nasa_eonet_analysis", ssl_context=ctx)

    for idx, row in top_coords.iterrows():
        location = geolocator.reverse(f"{row['lat_rounded']}, {row['lon_rounded']}")
        top_coords.at[idx, "region"] = location.address if location else "Unknown"
        time.sleep(1)

    top_coords["state"] = top_coords["region"].apply(lambda x: x.split(",")[-2].strip())
    top_coords = top_coords[~top_coords["state"].str.isdigit()]
    top_coords.to_csv(GEOCODED_CSV, index=False)

state_counts = (
    top_coords.groupby("state")["count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    state_counts,
    x="state", y="count",
    title="US States by Wildfire Event Count",
    labels={"state": "State", "count": "Number of Wildfire Events"}
)
fig.show()